# ASR Metrics Evaluation Showcase
Ten notatnik demonstruje wykorzystanie zoptymalizowanego modułu `metrics.py` do kompleksowej oceny systemu ASR.

In [ ]:
!pip install -r requirements.txt

## 1. Import i przygotowanie danych
Generujemy przykładowe dane wejściowe dla metryk. W prawdziwym scenariuszu użyłbyś wyników ze swojego systemu ASR.

In [ ]:
import numpy as np
import soundfile as sf
import time
import torch
from metrics import eval_all_metrics, preload_models

# Sprawdzenie dostępności GPU (CUDA)
if torch.cuda.is_available():
    print("GPU (CUDA) jest dostępne!")
    print(f"Nazwa urządzenia GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU (CUDA) NIE jest dostępne. Będzie używane CPU.")

# Generowanie plików testowych audio (1-sekundowy sygnał sinus)
sample_rate = 16000
t = np.linspace(0, 1.0, sample_rate)
sf.write('test_ref.flac', np.sin(2 * np.pi * 440. * t), sample_rate)
sf.write('test_hyp.flac', np.sin(2 * np.pi * 440. * t + 0.1), sample_rate)

ytrue_batch = ["the quick brown fox jumps over the lazy dog"]
ypred_batch = ["the quick brown fox jumps over the dog"]
ref_audios = ['test_ref.flac']
hyp_audios = ['test_hyp.flac']
decode_times = [0.5]  # Przykładowy czas trwania inferencji Twojego modelu ASR
encode_times = [0.2]

## 2. Pre-loading modeli (SpeechBrain & SeMaScore)
Aby pominąć czas ładowania wag z dysku do GPU podczas samej inferencji, ładujemy je z wyprzedzeniem.

In [ ]:
print("Ładowanie modeli do pamięci...")
preload_models()

## 3. Wywołanie ewaluacji
Uruchamiamy główną metodę ewaluacji i mierzymy czysty czas wyliczania metryk w oparciu o batch.

In [ ]:
import time

print("Trwa ewaluacja dla różnych rozmiarów batcha...")

batch_sizes = [8, 16, 32] # Lista rozmiarów batcha do przetestowania

# Oryginalne przykładowe dane, które będą powielane
original_ytrue = "the quick brown fox jumps over the lazy dog"
original_ypred = "the quick brown fox jumps over the dog"
original_ref_audio = 'test_ref.flac'
original_hyp_audio = 'test_hyp.flac'
original_decode_time = 0.5
original_encode_time = 0.2

for batch_size in batch_sizes:
    print(f"\n{'='*50}")
    print(f"Ewaluacja dla batch_size: {batch_size}")
    print(f"{'='*50}")

    # Generowanie danych dla bieżącego batch_size
    current_ytrue_batch = [original_ytrue] * batch_size
    current_ypred_batch = [original_ypred] * batch_size
    current_ref_audios = [original_ref_audio] * batch_size
    current_hyp_audios = [original_hyp_audio] * batch_size
    current_decode_times = [original_decode_time] * batch_size
    current_encode_times = [original_encode_time] * batch_size

    start_t = time.time()

    agg_res, ind_res = eval_all_metrics(
        ytrue_texts=current_ytrue_batch,
        ypred_texts=current_ypred_batch,
        ytrue_audio_paths=current_ref_audios,
        ypred_audio_paths=current_hyp_audios,
        decode_times=current_decode_times,
        encode_times=current_encode_times,
        log=False
    )

    eval_duration = time.time() - start_t

    # Obliczanie rzeczywistego RTF ewaluacji
    total_audio_duration_in_batch = batch_size * agg_res.get('audio_duration', 1.0) # Używamy 1.0 jako domyślnej długości, jeśli nie ma w agg_res
    evaluation_rtf = eval_duration / total_audio_duration_in_batch

    print(f"WYNIKI EWALUACJI (AGGREGATE) dla batch_size {batch_size}")
    for metric_name, value in agg_res.items():
        if isinstance(value, float):
            print(f'{metric_name.upper():<20}: {value:.4f}')
        else:
            print(f'{metric_name.upper():<20}: {value}')
    print(f'Czas wyliczania wszystkich metryk (cała inferencja) dla batch_size {batch_size}: {eval_duration:.4f} sekund')
    print(f'RTFX (ASR model, z wejścia) dla batch_size {batch_size}: {agg_res.get("rtfx", "N/A"):.4f}')
    print(f'Rzeczywisty RTF ewaluacji (cała procedura) dla batch_size {batch_size}: {evaluation_rtf:.4f}')

print('\n' + '='*50)
print('Zakończono ewaluację dla wszystkich rozmiarów batcha.')
print('='*50)